# Domain 5: Context Management & Reliability
## Task Statement 5.1: Manage Conversation Context to Preserve Critical Information
**Certification Weight**: 17% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to preserve transactional integrity across long multi-turn interactions using the **Case Facts** layer, avoid the catastrophic loss of numerical facts from **progressive summarization**, trim tool outputs, and combat the **lost-in-the-middle** effect.

---

## 🏗️ Architecture: Dual-Layer Context Preservation

```mermaid
flowchart TD
    UserTurn["Incoming User Turn"] --> ContextEngine["Context Composition Engine"]
    
    subgraph CaseFacts["Dedicated Case Facts Layer (Immutable)"]
        Facts["- Customer ID: CUST-8831<br>- Order Number: ORD-2026-99<br>- Refund Amount: $450.00<br>- Verified Identity: True"]
    end
    
    subgraph ConversationHistory["Conversation History Layer (Mutable)"]
        TurnSummary["Progressive Summary of past 15 turns<br>(Narrative dialog, troubleshooting attempts)"]
        RecentTurns["Last 2 turns in full verbatim detail"]
    end
    
    subgraph ToolTrimmer["Tool Normalization (PostToolUse)"]
        RawDB["Raw Database Payload (45 fields)"] --> TrimmedResult["Trimmed Payload (4 fields only)"]
    end
    
    CaseFacts --> ContextEngine
    ConversationHistory --> ContextEngine
    TrimmedResult --> ContextEngine
    
    ContextEngine --> ClaudePrompt["Constructed Prompt to Claude<br>(Case Facts at TOP to eliminate 'lost-in-the-middle')"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Context Management Technique | Correct Implementation | Exam Trap to Reject |
|---|---|---|
| **Transactional State Storage** | **Case Facts Pattern**: Store order IDs, refund dollar amounts, and customer IDs in an immutable structured block outside conversation summaries. | **TRAP**: Relying solely on recursive LLM summarization. LLM summarizers reliably drop or distort exact numbers, timestamps, and order strings over long threads. |
| **Tool Payload Optimization** | **Trim Tool Outputs**: Strip unneeded internal metadata fields before saving tool results into conversation history. | **TRAP**: Dumping raw 50-field enterprise database rows into history, filling the context window with useless noise. |
| **Combating 'Lost-in-the-Middle'** | Place critical system instructions, schemas, and Case Facts at the **top** or **bottom** of the context. | **TRAP**: Burying critical policy rules in the middle of a 150,000-token conversation history. |

---

## 💡 Architectural Decision Matrix

| Strategy | Token Efficiency | Fact Retention Fidelity | Best Used For |
|---|---|---|---|
| **Raw Conversation History** | Low (rapidly exhausts context) | 100% (until overflow) | Short, 2–3 turn quick workflows. |
| **Naive Progressive Summarization** | High | Low (hallucinates/drops numbers) | Casual conversational bots (NOT financial/billing). |
| **Case Facts + Rolling Summary** | High | **100% Deterministic** | Enterprise billing, customer support, legal workflows. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.1 Context Preservation.")

### ✂️ Step 2: Output Trimmer & Case Facts Manager
We implement the tool output trimmer and the persistent `CaseFacts` state container.

In [ ]:
# Step 2: Trimming and Persistent Facts Logic
RAW_ORDER_PAYLOAD = {
    "order_id": "ORD-2026-9921",
    "customer_id": "CUST-441",
    "status": "delivered",
    "amount": 149.99,
    "internal_db_primary_key": 998124,
    "payment_gateway_auth_token": "tok_live_secret_auth_99182371",
    "carrier_tracking_code": "TRACK_FEDEX_9981247182",
    "warehouse_rack_id": "RACK-4B-SHELF-2",
    "server_processing_node": "us-east-1a-worker-04",
}

def trim_order_output(raw_order: Dict[str, Any]) -> Dict[str, Any]:
    """Strips bulky internal telemetry, leaving only essential business fields."""
    return {
        "order_id": raw_order.get("order_id"),
        "customer_id": raw_order.get("customer_id"),
        "status": raw_order.get("status"),
        "amount": raw_order.get("amount"),
    }


@dataclass
class CaseFacts:
    customer_id: str
    customer_email: str
    order_id: Optional[str] = None
    order_amount: Optional[float] = None
    status: str = "Active"

    def render_prompt_block(self) -> str:
        """Renders persistent state block placed at the top of prompt context."""
        lines = [
            "### PERSISTENT CASE FACTS (IMMUTABLE STATE)",
            f"- Customer ID: {self.customer_id}",
            f"- Customer Email: {self.customer_email}",
        ]
        if self.order_id:
            lines.append(f"- Active Order: {self.order_id} (${self.order_amount:.2f})")
        lines.append(f"- Status: {self.status}")
        return "\n".join(lines)


def lookup_order_tool(order_id: str, case_facts: CaseFacts) -> Dict[str, Any]:
    """Simulates tool execution that updates persistent CaseFacts state."""
    trimmed = trim_order_output(RAW_ORDER_PAYLOAD)
    case_facts.order_id = trimmed["order_id"]
    case_facts.order_amount = trimmed["amount"]
    return trimmed

print(" Output trimmer and CaseFacts container ready.")

### 🧪 Step 3: Interactive Simulation
1. Compare raw payload size vs trimmed payload size.
2. Observe how tool execution updates the persistent Case Facts block.

In [ ]:
# Step 3: Simulation
# 1. Trimming comparison
trimmed_order = trim_order_output(RAW_ORDER_PAYLOAD)
raw_size = len(json.dumps(RAW_ORDER_PAYLOAD))
trimmed_size = len(json.dumps(trimmed_order))

print(f"Raw Payload Characters: {raw_size} chars")
print(f"Trimmed Payload Characters: {trimmed_size} chars")
print(f"Token Footprint Reduction: ~{((raw_size - trimmed_size) / raw_size) * 100:.1f}%")

# 2. CaseFacts persistence across turns
facts = CaseFacts(customer_id="CUST-441", customer_email="alice@test.com")
print("\nInitial Prompt Block:\n", facts.render_prompt_block())

# Tool execution updates facts
lookup_order_tool("ORD-2026-9921", facts)
print("\nUpdated Prompt Block after Order Lookup:\n", facts.render_prompt_block())

### 🔬 Step 4: Verification Assertions
We assert that:
1. Sensitive internal keys (rack ID, gateway token) are stripped from trimmed output.
2. CaseFacts reflects customer and order state reliably.

In [ ]:
# Step 4: Verification Checks
assert "warehouse_rack_id" not in trimmed_order
assert "payment_gateway_auth_token" not in trimmed_order
assert trimmed_order["order_id"] == "ORD-2026-9921"
assert trimmed_order["amount"] == 149.99

rendered = facts.render_prompt_block()
assert "CUST-441" in rendered
assert "ORD-2026-9921" in rendered
assert "$149.99" in rendered

print(" All verification assertions passed! Context preservation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Trim tool outputs to essential fields before appending them to conversation history.
- **Rule 2**: Maintain critical identifiers in a persistent **CaseFacts** block outside conversation summaries.
- **Rule 3**: Strip raw backend telemetry and internal keys to conserve context window capacity.